In [ ]:
!pip install selenium
!pip install bs4
!pip install numpy
!pip install pandas

import sys

from selenium import webdriver
from bs4 import BeautifulSoup
import time
import numpy as np
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
import pandas as pd
sys.path.insert(0,"C:\\Users\jurge\\Documents\\Selenium")

In [3]:
all_articles = []
page = 1
article_title = []
full_data = pd.DataFrame()

driver = webdriver.Chrome(service=service, options=options)
driver.get(url=url)
all_articles.append(driver.find_element(By.XPATH,'//article[@data-test]'))
for article in all_articles:
    article_date = article.text.strip()
    print(article_date)


NameError: name 'service' is not defined

In [ ]:
#!/usr/bin/python3
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import csv

url = 'https://www.investing.com/equities/amazon-com-inc-news'
path = "C:\\Users\\jurge\\Documents\\Selenium\\chromedriver.exe"

def start_driver():
    options = Options()
    # options.add_argument('--headless')  # Uncomment this after testing
    options.add_experimental_option('detach', True)
    options.add_argument('--disable-blink-features=AutomationControlled')
    return webdriver.Chrome(service=Service(path), options=options)

# Initialize main driver
page_counter = 1



while page_counter <= 1000: 
    driver = start_driver()
    if page_counter == 1:
        driver.get(f"{url}")
    elif page_counter > 1:
        driver.get(f"{url}/{page_counter}")
    time.sleep(2)


    # Prepare CSV file
    with open('article_data.csv', mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Title", "URL", "Content"])  # CSV header
        
        # Scrape the page for article links
        articles = driver.find_elements(By.XPATH, "//article[@data-test]")
        for article in articles:
            try:
                link_element = article.find_element(By.TAG_NAME, "a")
                article_url = link_element.get_attribute("href")
                #title = link_element.get_attribute("title").strip()  # Ensure title is captured
                #date = article.find_element(By.XPATH, ".//time").text.strip()  # Ensure date is captured
                title = article.text.strip()

                # Open a new driver for the article page
                article_driver = start_driver()
                article_driver.get(article_url)
                
                # Wait for content to load
                WebDriverWait(article_driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.article_WYSIWYG__O0uhw"))
                )

                # Get page source and parse with BeautifulSoup
                soup = BeautifulSoup(article_driver.page_source, "html.parser")

                # Extract article content
                content_div = soup.select_one("div.article_WYSIWYG__O0uhw")
                content = content_div.get_text(separator="\n").strip() if content_div else ""

                # Write data to CSV
                writer.writerow([title, article_url, content])

                print(f"Scraped: {title}")

                # Close article driver
                article_driver.quit()

            except Exception as e:
                print(f"Skipped article due to error: {e}")
                continue

            finally:
                if article_driver:
                    article_driver.quit()
    page_counter +=1
        
    driver.quit()